## Assemble UMAP coordinates with raw counts

In the previous notebook, UMAP coordinates were generated. However, this resulted in normalized values stored without the original raw counts.

Here, we'll pull the raw count data and add the `X_umap` and the updated metadata field, `vaccine.year`, to the version of the data with raw counts.

In [1]:
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc

In [2]:
if not os.path.isdir('output'):
    os.makedirs('output')

### Helper functions

In [3]:
def find_stored_files(search_id, store = 'Service_Core'):
    ps_df = hisepy.list_files_in_project_store(store)
    ps_df = ps_df[['id', 'name']]
    search_df = ps_df[ps_df['name'].str.contains(search_id)]
    
    return search_df

In [4]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

In [5]:
def add_colorsets(adata, color_files):
    color_dicts = {}
    for level, file in color_files.items():
        color_df = pd.read_csv(file)

        color_dict = dict(zip(color_df[level], color_df[f'{level}_color']))
        color_dicts[level] = color_dict

    for level, color_dict in color_dicts.items():
        color_order = adata.obs[level].cat.categories
        level_colors = [color_dict[x] for x in color_order]
        adata.uns[f'{level}_colors'] = level_colors
        
    return adata

### Datasets

#### Raw count data

In [6]:
search_id = 'boron-potassium-aluminum'

In [7]:
search_df = find_stored_files(search_id, 'cohorts')

In [8]:
search_df

,id,name
17360,65b9c639-8bac-485d-9cdf-ec66acfd5869,boron-potassium-aluminum/SoundLife_YoungAdult_...
17361,5b7df27d-9957-4e3d-a054-328f2e224304,boron-potassium-aluminum/SoundLife_YoungAdult_...
17362,bc012978-7fa9-466e-9cac-f61fc67d9431,boron-potassium-aluminum/SoundLife_YoungAdult_...
17363,c2e6d243-8dd6-4e8b-9eda-62961896893e,boron-potassium-aluminum/SoundLife_YoungAdult_...
17364,5735fc28-f699-4d56-be49-4bb4bff39ed1,boron-potassium-aluminum/SoundLife_OlderAdult_...
17365,3b7c5d92-0a02-4acd-a743-a01f54cf5d6e,boron-potassium-aluminum/SoundLife_OlderAdult_...
17366,2e20043b-7e1a-4e33-88d8-50134db78ce6,boron-potassium-aluminum/SoundLife_OlderAdult_...
17367,4250a2b1-0eae-42c6-9cc6-5e6ebb83764d,boron-potassium-aluminum/SoundLife_OlderAdult_...


In [9]:
raw_uuids = dict(zip([os.path.basename(n) for n in search_df['name']], search_df['id']))

In [10]:
raw_uuids

{'SoundLife_YoungAdult_Female_CMVneg.h5ad': '65b9c639-8bac-485d-9cdf-ec66acfd5869',
 'SoundLife_YoungAdult_Female_CMVpos.h5ad': '5b7df27d-9957-4e3d-a054-328f2e224304',
 'SoundLife_YoungAdult_Male_CMVneg.h5ad': 'bc012978-7fa9-466e-9cac-f61fc67d9431',
 'SoundLife_YoungAdult_Male_CMVpos.h5ad': 'c2e6d243-8dd6-4e8b-9eda-62961896893e',
 'SoundLife_OlderAdult_Female_CMVneg.h5ad': '5735fc28-f699-4d56-be49-4bb4bff39ed1',
 'SoundLife_OlderAdult_Female_CMVpos.h5ad': '3b7c5d92-0a02-4acd-a743-a01f54cf5d6e',
 'SoundLife_OlderAdult_Male_CMVneg.h5ad': '2e20043b-7e1a-4e33-88d8-50134db78ce6',
 'SoundLife_OlderAdult_Male_CMVpos.h5ad': '4250a2b1-0eae-42c6-9cc6-5e6ebb83764d'}

#### Data with UMAP coordinates

In [11]:
search_id = 'barium-samarium-thallium'

In [12]:
search_df = find_stored_files(search_id, 'cohorts')

In [13]:
search_df

,id,name
17766,bdf86b7b-da55-42cb-8a9a-2f1ff29723ad,barium-samarium-thallium/SoundLife_YoungAdult_...
17767,32e96ab1-29af-4373-93d8-6824ed8de11e,barium-samarium-thallium/SoundLife_YoungAdult_...
17768,7e6130fd-1a70-4fc4-9c34-162b85fe3018,barium-samarium-thallium/SoundLife_YoungAdult_...
17769,047f7094-5b68-4f72-8e98-1f9e878f9ea3,barium-samarium-thallium/SoundLife_YoungAdult_...
17770,0d0273de-bebc-4652-b97b-53b0f470f8d3,barium-samarium-thallium/SoundLife_OlderAdult_...
17771,9e41c22f-561a-4a3b-89c5-1f94d1c268a0,barium-samarium-thallium/SoundLife_OlderAdult_...
17772,bde6ec90-eb77-4d31-bcea-aa78cd1f257f,barium-samarium-thallium/SoundLife_OlderAdult_...
17773,12d35e6e-50f1-4766-89c2-66c4f7d3951f,barium-samarium-thallium/SoundLife_OlderAdult_...


In [14]:
umap_uuids = dict(zip([os.path.basename(n) for n in search_df['name']], search_df['id']))

In [15]:
umap_uuids

{'SoundLife_YoungAdult_Female_CMVneg.h5ad': 'bdf86b7b-da55-42cb-8a9a-2f1ff29723ad',
 'SoundLife_YoungAdult_Female_CMVpos.h5ad': '32e96ab1-29af-4373-93d8-6824ed8de11e',
 'SoundLife_YoungAdult_Male_CMVneg.h5ad': '7e6130fd-1a70-4fc4-9c34-162b85fe3018',
 'SoundLife_YoungAdult_Male_CMVpos.h5ad': '047f7094-5b68-4f72-8e98-1f9e878f9ea3',
 'SoundLife_OlderAdult_Female_CMVneg.h5ad': '0d0273de-bebc-4652-b97b-53b0f470f8d3',
 'SoundLife_OlderAdult_Female_CMVpos.h5ad': '9e41c22f-561a-4a3b-89c5-1f94d1c268a0',
 'SoundLife_OlderAdult_Male_CMVneg.h5ad': 'bde6ec90-eb77-4d31-bcea-aa78cd1f257f',
 'SoundLife_OlderAdult_Male_CMVpos.h5ad': '12d35e6e-50f1-4766-89c2-66c4f7d3951f'}

### Load raw data and transfer UMAP values

In [16]:
for dataset in raw_uuids.keys():
    out_file = 'output/' + dataset

    if not os.path.isfile(out_file):
        raw_file = hisepy.cache_files([raw_uuids[dataset]])[0]
        umap_file = hisepy.cache_files([umap_uuids[dataset]])[0]
    
        adata = sc.read_h5ad(raw_file)
        umap_adata = sc.read_h5ad(umap_file, backed = 'r')
    
        # Copy UMAP
        adata.obsm['X_umap'] = umap_adata.obsm['X_umap']
        adata.uns['umap'] = umap_adata.uns['umap']
        # Copy vaccine.year
        adata.obs['vaccine_year'] = umap_adata.obs['vaccine.year']
        # Copy colors
        adata.uns['AIFI_L1_colors'] = umap_adata.uns['AIFI_L1_colors']
        adata.uns['AIFI_L2_colors'] = umap_adata.uns['AIFI_L2_colors']
        adata.uns['AIFI_L3_colors'] = umap_adata.uns['AIFI_L3_colors']
    
        # Save updated version
        adata.write_h5ad(out_file)

        del(adata)

## Upload data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for distribution.

In [17]:
study_space_uuid = 'de025812-5e73-4b3c-9c3b-6d0eac412f2a'
title = 'Sound Life Subject Group .h5ad with UMAP and vaccine.year {d}'.format(d = date.today())

In [18]:
search_id = element_id()
search_id

'copernicium-scandium-copernicium'

In [19]:
in_files = list(raw_uuids.values()) + list(umap_uuids.values())
in_files

['65b9c639-8bac-485d-9cdf-ec66acfd5869',
 '5b7df27d-9957-4e3d-a054-328f2e224304',
 'bc012978-7fa9-466e-9cac-f61fc67d9431',
 'c2e6d243-8dd6-4e8b-9eda-62961896893e',
 '5735fc28-f699-4d56-be49-4bb4bff39ed1',
 '3b7c5d92-0a02-4acd-a743-a01f54cf5d6e',
 '2e20043b-7e1a-4e33-88d8-50134db78ce6',
 '4250a2b1-0eae-42c6-9cc6-5e6ebb83764d',
 'bdf86b7b-da55-42cb-8a9a-2f1ff29723ad',
 '32e96ab1-29af-4373-93d8-6824ed8de11e',
 '7e6130fd-1a70-4fc4-9c34-162b85fe3018',
 '047f7094-5b68-4f72-8e98-1f9e878f9ea3',
 '0d0273de-bebc-4652-b97b-53b0f470f8d3',
 '9e41c22f-561a-4a3b-89c5-1f94d1c268a0',
 'bde6ec90-eb77-4d31-bcea-aa78cd1f257f',
 '12d35e6e-50f1-4766-89c2-66c4f7d3951f']

In [20]:
out_files = ['output/' + f for f in raw_uuids.keys()]
out_files

['output/SoundLife_YoungAdult_Female_CMVneg.h5ad',
 'output/SoundLife_YoungAdult_Female_CMVpos.h5ad',
 'output/SoundLife_YoungAdult_Male_CMVneg.h5ad',
 'output/SoundLife_YoungAdult_Male_CMVpos.h5ad',
 'output/SoundLife_OlderAdult_Female_CMVneg.h5ad',
 'output/SoundLife_OlderAdult_Female_CMVpos.h5ad',
 'output/SoundLife_OlderAdult_Male_CMVneg.h5ad',
 'output/SoundLife_OlderAdult_Male_CMVpos.h5ad']

In [21]:
len(out_files)

8

In [22]:
hisepy.upload.upload_files(
    files = out_files,
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

checking if conda environment can compile...


{'Message': 'General Okay-ness',
 'VisualizationId': '00000000-0000-0000-0000-000000000000',
 'AbstractionId': '00000000-0000-0000-0000-000000000000',
 'TraceId': 'fb1b2aa1-c5c7-46c7-90d6-0828dbed731b',
 'ProcessId': '2e70108f-0542-49b1-984a-a1e8421dc7cb',
 'WorkflowId': 'abad90eb-4832-4ba7-a23e-3134b6763e9f',
 'FileIds': ['b603ddf4-e792-4173-9436-e24240beb68a',
  '5b461057-3ae1-4747-b036-496930ccae55',
  '82d3e250-3449-48ad-a9c1-e257ad2de253',
  '9cf3b267-e412-4427-af37-be43b1a7ffb0',
  '5a6d47f8-0296-4d18-a61d-c8595946d0d9',
  '28727bd7-c3d3-460b-9159-0e67e2cebacf',
  '9aa081ab-173f-499d-b1c2-155978071e12',
  '4e818408-a623-4bbf-ad9b-b10fe03ad7ad']}

In [23]:
import session_info
session_info.show()